In [4]:
import re
import math
import json
from collections import defaultdict
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

# -------------------- Utility Functions --------------------

def load_stopwords(file_path):
    with open(file_path, 'r') as f:
        return {line.strip() for line in f}

def tokenize(text, stop_words):
    tokens = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    return [token for token in tokens if token not in stop_words]

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

def load_forward_index(file_path):
    forward_index = {}
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split(':', 1)
            doc_id = int(parts[0])
            word_freqs = parts[1].strip().split(';')
            forward_index[doc_id] = {wf.split(':')[0].strip(): int(wf.split(':')[1]) for wf in word_freqs if wf}
    return forward_index

def load_inverted_index(file_path):
    inverted_index = defaultdict(dict)
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split(':', 1)
            word = parts[0].strip()
            doc_freqs = parts[1].strip().split(';')
            inverted_index[word] = {int(df.split(':')[0]): int(df.split(':')[1]) for df in doc_freqs if df}
    return inverted_index

def load_dictionaries(word_dict_path, doc_dict_path):
    with open(word_dict_path, 'r') as f:
        word_dict = json.load(f)
    with open(doc_dict_path, 'r') as f:
        doc_dict = json.load(f)
    return word_dict, doc_dict

def compute_tf_idf(forward_index, inverted_index, total_docs):
    tf_idf = defaultdict(dict)
    idf = {}
    for word_id, doc_freqs in inverted_index.items():
        idf[word_id] = math.log10(total_docs / len(doc_freqs))
        for doc_id, tf in doc_freqs.items():
            tf_idf[doc_id][word_id] = (1 + math.log10(tf)) * idf[word_id]
    return tf_idf, idf

def cosine_similarity(query_vector, doc_vector):
    dot_product = sum(query_vector[word] * doc_vector.get(word, 0) for word in query_vector)
    query_mag = math.sqrt(sum(v ** 2 for v in query_vector.values()))
    doc_mag = math.sqrt(sum(v ** 2 for v in doc_vector.values()))
    return dot_product / (query_mag * doc_mag) if query_mag and doc_mag else 0

def load_queries(query_file):
    queries = []
    with open(query_file, 'r') as f:
        data = f.read()
        topics = re.findall(r'<top>(.*?)</top>', data, re.DOTALL)
        for topic in topics:
            num = int(re.search(r'<num> Number: (\d+)', topic).group(1))
            title = re.search(r'<title>(.*?)\n', topic).group(1).strip()
            desc = re.search(r'<desc> Description:(.*?)\n', topic, re.DOTALL).group(1).strip()
            narr = re.search(r'<narr> Narrative:(.*?)$', topic, re.DOTALL).group(1).strip()
            queries.append((num, title, desc, narr))
    return queries

# -------------------- Core Query Processing --------------------

def process_query(query_text, idf, tf_idf, stop_words, word_dict):
    tokens = tokenize(query_text, stop_words)
    stemmed = stem_tokens(tokens)
    query_vector = {}

    for token in stemmed:
        token_id = word_dict.get(token)
        if token_id is not None:
            token_id_str = str(token_id)
            if token_id_str in idf:
                query_vector[token_id_str] = query_vector.get(token_id_str, 0) + 1

    for word_id in query_vector:
        query_vector[word_id] = (1 + math.log10(query_vector[word_id])) * idf.get(word_id, 0)

    scores = [(doc_id, cosine_similarity(query_vector, doc_vector)) for doc_id, doc_vector in tf_idf.items()]
    return sorted([s for s in scores if s[1] > 0], key=lambda x: x[1], reverse=True)

# -------------------- Main Execution --------------------

stop_words = load_stopwords('stopwordlist.txt')
forward_index = load_forward_index('forward_index.txt')
inverted_index = load_inverted_index('inverted_index.txt')
word_dict, doc_dict = load_dictionaries('word_dict.json', 'doc_dict.json')

total_docs = len(forward_index)
tf_idf, idf = compute_tf_idf(forward_index, inverted_index, total_docs)
queries = load_queries('topics.txt')
id_to_doc = {v: k for k, v in doc_dict.items()}

query_settings = {
    "title": lambda q: q[1],
    "title_description": lambda q: q[1] + " " + q[2],
    "title_narration": lambda q: q[1] + " " + q[3],
}

with open('vsm_output.txt', 'w') as output_file:
    for query_num, title, desc, narr in queries:
        for setting_name, extractor in query_settings.items():
            query_text = extractor((query_num, title, desc, narr))
            scores = process_query(query_text, idf, tf_idf, stop_words, word_dict)
            for rank, (doc_id, score) in enumerate(scores[:1000], start=1):
                doc_name = id_to_doc.get(str(doc_id), f"Doc_{doc_id}")
                output_file.write(f"{query_num:<10}{doc_name:<30}{rank:<10}{score:.6f}\n")
            print(f"Query {query_num} [{setting_name}] completed.")

print("All queries processed. Results saved to 'vsm_output.txt'.")


Query 352 [title] completed.
Query 352 [title_description] completed.
Query 352 [title_narration] completed.
Query 353 [title] completed.
Query 353 [title_description] completed.
Query 353 [title_narration] completed.
Query 354 [title] completed.
Query 354 [title_description] completed.
Query 354 [title_narration] completed.
Query 359 [title] completed.
Query 359 [title_description] completed.
Query 359 [title_narration] completed.
All queries processed. Results saved to 'vsm_output.txt'.
